In [1]:
import torch
import os
import pandas as pd

def read_csv(csv_path):
    return pd.read_csv(csv_path)

In [2]:
if os.name == 'nt':
    csv_path = r'Z:\EgoMocap\work\HumanML3D\index.csv'
else:
    csv_path = '/CT/EgoMocap/work/HumanML3D/index.csv'

df = read_csv(csv_path)
print(len(df))
print(df.loc[0])

14616
source_path    ./pose_data/KIT/3/kick_high_left02_poses.npy
start_frame                                               0
end_frame                                               117
new_name                                         000000.npy
Name: 0, dtype: object


In [3]:
path_dict = dict()

for i in range(len(df)):
    original_path = df.loc[i]['source_path'][12: -3] + 'npz'
    start_frame = df.loc[i]['start_frame']
    end_frame = df.loc[i]['end_frame']
    new_name = df.loc[i]['new_name']
    # merge with original path, list of start and end frame
    if original_path in path_dict.keys():
        path_dict[original_path].append({'start_frame': start_frame, 'end_frame': end_frame, 'humanml3d_name': new_name})
    else:
        path_dict[original_path] = [{'start_frame': start_frame, 'end_frame': end_frame, 'humanml3d_name': new_name}]

In [6]:
max_len = 0

for k, v in path_dict.items():
    max_len = max(max_len, v[-1]['end_frame'] - v[-1]['start_frame'])
print(max_len)
    

200


In [5]:
import pickle

# save the information to a pkl file
save_path = r'Z:\EgoMocap\work\EgoOmniMocap\scripts\path_dict.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(path_dict, f)

In [28]:
root_dir = r'\\winfs-inf\CT\EgoMocap\work\IMUPoser\data\processed_imuposer\AMASS_w_name'
amass_seq_name_list = os.listdir(root_dir)
print(amass_seq_name_list)

['ACCAD', 'BioMotionLab_NTroje', 'BMLhandball', 'BMLmovi', 'CMU', 'DanceDB', 'DFaust_67', 'Eyes_Japan_Dataset', 'HUMAN4D', 'HumanEva', 'MPI_HDM05', 'MPI_Limits', 'MPI_mosh', 'SFU', 'SSM_synced', 'TCD_handMocap', 'TotalCapture', 'Transitions_mocap']


In [30]:
# load the cmu data for test
for amass_seq_name in amass_seq_name_list:
    print('processing {}'.format(amass_seq_name))
    name_path = r'Z:\EgoMocap\work\IMUPoser\data\processed_imuposer\AMASS_w_name\{}\name.pt'.format(amass_seq_name)
    joint_path = r'Z:\EgoMocap\work\IMUPoser\data\processed_imuposer\AMASS_w_name\{}\joint.pt'.format(amass_seq_name)
    length_path = r'Z:\EgoMocap\work\IMUPoser\data\processed_imuposer\AMASS_w_name\{}\length.pt'.format(amass_seq_name)
    pose_path = r'Z:\EgoMocap\work\IMUPoser\data\processed_imuposer\AMASS_w_name\{}\pose.pt'.format(amass_seq_name)
    shape_path = r'Z:\EgoMocap\work\IMUPoser\data\processed_imuposer\AMASS_w_name\{}\shape.pt'.format(amass_seq_name)
    tran_path = r'Z:\EgoMocap\work\IMUPoser\data\processed_imuposer\AMASS_w_name\{}\tran.pt'.format(amass_seq_name)
    vacc_path = r'Z:\EgoMocap\work\IMUPoser\data\processed_imuposer\AMASS_w_name\{}\vacc.pt'.format(amass_seq_name)
    vrot_path = r'Z:\EgoMocap\work\IMUPoser\data\processed_imuposer\AMASS_w_name\{}\vrot.pt'.format(amass_seq_name)
    name_data = torch.load(name_path)
    joint_data = torch.load(joint_path)
    length_data = torch.load(length_path)
    pose_data = torch.load(pose_path)
    shape_data = torch.load(shape_path)
    tran_data = torch.load(tran_path)
    vacc_data = torch.load(vacc_path)
    vrot_data = torch.load(vrot_path)
    
    data_dict_list = []
    
    for i, name in enumerate(name_data):
        name_l = name.split('/')[-3:]
        imuposer_name = '/'.join(name_l)
        if imuposer_name in path_dict.keys():
            print(imuposer_name)
            data_dict = {
                'name': imuposer_name,
                'joint': joint_data[i],
                'length': length_data[i],
                'pose': pose_data[i],
                'shape': shape_data[i],
                'tran': tran_data[i],
                'vacc': vacc_data[i],
                'vrot': vrot_data[i],
                'humanml3d': path_dict[imuposer_name]
            }
            data_dict_list.append(data_dict)
    # save
    save_path = r'Z:\EgoMocap\work\EgoOmniMocap\scripts\amass_data_dict\{}.pt'.format(amass_seq_name)
    torch.save(data_dict_list, save_path)

processing ACCAD
ACCAD/s007/QkWalk1_poses.npz
ACCAD/Male1Running_c3d/Run C24 - quick side step left_poses.npz
ACCAD/Male1Running_c3d/Run C25 - quick side step right_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E15 - block left middle_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E19 - dodge left t2_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E12 - block left low_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E22 - duck right_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E2 - Turn around left_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E17 - block middle high_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E16 - block right middle_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E19 - dodge left_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E8 - bounce_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E21 - duck left_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E13 - block right high_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E11 - block left high_poses.npz
ACCAD/MartialArtsWalksTurns_c3d/E5 - retreat_poses.npz
ACCAD/MartialArtsWalksT